# Cross-modal consistency and ablations

This standalone notebook trains the emotion and learned-consistency ablations and separately measures CLIP claim-image similarity. Similarity is an analysis signal, not factual verification. Use fixed repository splits and keep `SMOKE_TEST=True` for a first run.

In [ ]:
%pip -q install -U torch transformers accelerate pandas numpy scikit-learn pillow pyyaml tqdm matplotlib seaborn

In [ ]:
import os, sys
from pathlib import Path
import torch

SMOKE_TEST = True
SMOKE_SIZE = 32
REPO_ROOT_OVERRIDE = ""
candidates = [Path(REPO_ROOT_OVERRIDE)] if REPO_ROOT_OVERRIDE else []
candidates += [Path("/kaggle/working/FakeNewsDetector"), Path("/content/FakeNewsDetector"), Path.cwd()]
REPO_ROOT = next((p for p in candidates if (p / "src/experiments/run_experiment.py").exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Set REPO_ROOT_OVERRIDE to the uploaded repository root.")
os.environ["PYTHONPATH"] = str(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
OUTPUT_ROOT = Path("/kaggle/working/research_outputs" if Path("/kaggle").exists() else "/content/research_outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(REPO_ROOT, torch.cuda.is_available())

In [ ]:
import yaml
import pandas as pd
from src.experiments.run_experiment import run

CONFIGS = ["cross_attention_emotion.yaml", "cross_attention_consistency.yaml"]
results = []
for filename in CONFIGS:
    with (REPO_ROOT / "configs/experiments" / filename).open(encoding="utf-8") as handle:
        config = yaml.safe_load(handle)
    config["experiment"]["output_dir"] = str(OUTPUT_ROOT)
    config["experiment"]["name"] = Path(filename).stem + ("_smoke" if SMOKE_TEST else "")
    config["experiment"]["smoke_test"] = SMOKE_TEST
    config["experiment"]["smoke_size"] = SMOKE_SIZE
    runtime = OUTPUT_ROOT / (Path(filename).stem + "_runtime.yaml")
    runtime.write_text(yaml.safe_dump(config), encoding="utf-8")
    try:
        results.append(run(str(runtime)))
    except Exception as exc:
        print("Skipped/failed", filename, type(exc).__name__, exc)

ablation_table = pd.DataFrame([
    {"experiment": result["experiment"], **result["test_metrics"]}
    for result in results
])
ablation_table.to_csv(OUTPUT_ROOT / "ablation_results.csv", index=False)
ablation_table

In [ ]:
from PIL import Image
from src.experiments.consistency import CLIPConsistencyScorer

TEST_CSV = REPO_ROOT / "data/splits/test.csv"
IMAGE_DIR = REPO_ROOT / "data/raw/images"
test = pd.read_csv(TEST_CSV).head(SMOKE_SIZE if SMOKE_TEST else len(pd.read_csv(TEST_CSV)))
try:
    scorer = CLIPConsistencyScorer(device="cuda" if torch.cuda.is_available() else "cpu")
    scores = []
    for _, row in test.iterrows():
        image = Image.open(IMAGE_DIR / str(row["image_url"])).convert("RGB")
        scores.append(scorer.score(str(row["title"]), image))
    consistency = pd.DataFrame({"id": test["id"], "label": test["label"], "clip_similarity": scores})
    consistency.to_csv(OUTPUT_ROOT / "clip_consistency_scores.csv", index=False)
    display(consistency.describe(include="all"))
except Exception as exc:
    print("Optional CLIP consistency analysis unavailable:", type(exc).__name__, exc)

The emotion and consistency rows are ablations, not automatically improvements. Compare their held-out metrics against the main-model table only after full runs.

# Visual outputs

These plots compare ablations and show the distribution of CLIP consistency scores when that optional analysis completes.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
if not ablation_table.empty:
    metrics = [c for c in ["accuracy", "f1_macro", "roc_auc"] if c in ablation_table.columns]
    ax = ablation_table.set_index("experiment")[metrics].plot(kind="bar", figsize=(10, 5), ylim=(0, 1), rot=20)
    ax.set_ylabel("Score")
    ax.set_title("Emotion and consistency ablation comparison")
    plt.tight_layout()
    plt.savefig(OUTPUT_ROOT / "ablation_comparison.png", dpi=160)
    plt.show()

consistency_path = OUTPUT_ROOT / "clip_consistency_scores.csv"
if consistency_path.exists():
    consistency = pd.read_csv(consistency_path)
    plt.figure(figsize=(8, 5))
    sns.histplot(data=consistency, x="clip_similarity", hue="label", bins=20, kde=True, element="step")
    plt.title("CLIP claim-image similarity by label")
    plt.tight_layout()
    plt.savefig(OUTPUT_ROOT / "clip_consistency_distribution.png", dpi=160)
    plt.show()
else:
    print("No CLIP consistency CSV found; run the optional consistency cell first.")